# Chapter 7: GenAI recs. 

TODO: Get code from DataBricks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github//modern-recommender-systems/blob/main/notebooks/chapter-07/gen_recsys.ipynb)

This notebook introduces the basics of recommender systems.

In [2]:
# Cell 2: Environment Setup
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu


In [3]:

# One-line setup for Colab users
setup_colab_environment()

# Check GPU availability
check_gpu()

Not running in Colab, skipping setup.
⚠ No GPU detected.


False

In [5]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from recsys.data import loaders

DATA_PATH = get_data_path()

In [6]:
print("Ready to build recommender systems!")

Ready to build recommender systems!


In [ ]:
# load data loading and preprocessing here

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

# --- 1. PREPARE THE "TEXT" DATA ---
# We take the Semantic IDs from the previous step (Method 2)
# and convert them into "sentences".
# Format: "SID_0_1 SID_0_3 SID_1_0" (SID = Semantic ID)

def tuple_to_token(semantic_tuple):
    # Converts (0, 1) -> "SID_0_1"
    return f"SID_{semantic_tuple[0]}_{semantic_tuple[1]}"

# Create training file string
train_text = ""
for session in user_sessions:
    # Convert ["Star Wars", "Star Trek"] -> "SID_1_1 SID_1_2"
    token_sequence = [tuple_to_token(title_to_id[title]) for title in session]

    # Join into a sentence and add to training text
    # We add <|endoftext|> so the model knows where a session stops
    train_text += " ".join(token_sequence) + " <|endoftext|>\n"

# Save to a dummy file for the HuggingFace Dataset loader
with open("train_recsys.txt", "w") as f:
    f.write(train_text)

print("Sample Training Data:")
print(train_text.split('\n')[0])


# --- 2. LOAD THE FOUNDATION MODEL ---
model_name = "distilgpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# --- 3. EXPAND VOCABULARY ---
# Ideally, we add our new "SID_x_y" tokens to the tokenizer
# so the model treats them as single words, not split parts.
new_tokens = set()
for session in user_sessions:
    for title in session:
        new_tokens.add(tuple_to_token(title_to_id[title]))

# Add tokens and resize model embeddings
num_added_toks = tokenizer.add_tokens(list(new_tokens))
model.resize_token_embeddings(len(tokenizer))
print(f"Added {num_added_toks} new Semantic ID tokens to the vocabulary.")


# --- 4. THE TRAINING LOOP (Fine-Tuning) ---
# We use standard Causal Language Modeling (predict next token)
train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="train_recsys.txt",
    block_size=128
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False # mlm=False means "Predict Next Token" (GPT style)
)

training_args = TrainingArguments(
    output_dir="./recsys_gpt",
    overwrite_output_dir=True,
    num_train_epochs=10,       # In real life, use fewer epochs on more data
    per_device_train_batch_size=2,
    learning_rate=5e-4,        # Slightly higher LR for new vocab
    save_steps=500
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print("\nStarting Fine-Tuning...")
trainer.train()
print("Fine-Tuning Complete. The model now speaks 'RecSys'!")